# SIRA Cross-Model Transfer Test on Colab L4

This notebook tests whether the same SIRA attack pipeline transfers across three attack-model families:

- Llama 3.2 3B Instruct: released-code SIRA-Tiny checkpoint
- Google Gemma 2 2B IT: non-Llama family
- Qwen 2.5 7B Instruct: non-Llama family and larger model
- Cognitive Integrity Grid Masking: proposed deterministic 2x2 grid-guided baseline
- Normal Rewrite Control: same rewrite model without grid masking

Testing two non-Llama families can provide evidence of transferability, but it cannot prove SIRA works on every LLM. The grid baseline verifies an external token-state trace; it does not expose or verify hidden chain-of-thought.

Before running, choose **Runtime > Change runtime type > L4 GPU**. Add a Colab Secret named `HF_TOKEN` from a Hugging Face account that can download Llama 3.2 and Gemma 2.


In [ ]:
# Check that Colab assigned the requested L4 GPU.
import subprocess

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()

print("GPU:", gpu_name)
if "L4" not in gpu_name:
    raise RuntimeError("This notebook requires an L4 GPU. Change the Colab runtime type and try again.")


In [ ]:
# Experiment settings. Start with 10 samples, then increase after a successful run.
REPO_URL = "https://github.com/hanifnoerr/Self-information-Rewrite-Attack.git"
BRANCH = "codex/browser-colab-l4"
REPO_DIR = "/content/Self-information-Rewrite-Attack"
OUTPUT_ROOT = "/content/sira_outputs"

ALGORITHM = "KGW"
SAMPLES = 10
RESET_OUTPUTS = True

MODEL_RUNS = [
    {
        "label": "llama_3_2_3b",
        "display_name": "Llama 3.2 3B Instruct",
        "model_family": "Llama",
        "model_name": "meta-llama/Llama-3.2-3B-Instruct",
        "parameter_size": "3B",
        "quantization": "bf16",
        "load_in_4bit": False,
    },
    {
        "label": "gemma_2_2b",
        "display_name": "Gemma 2 2B IT",
        "model_family": "Gemma",
        "model_name": "google/gemma-2-2b-it",
        "parameter_size": "2B",
        "quantization": "bf16",
        "load_in_4bit": False,
    },
    {
        "label": "qwen_2_5_7b",
        "display_name": "Qwen 2.5 7B Instruct",
        "model_family": "Qwen",
        "model_name": "Qwen/Qwen2.5-7B-Instruct",
        "parameter_size": "7B",
        "quantization": "4-bit NF4 with bf16 compute",
        "load_in_4bit": True,
    },
]

print(f"Algorithm: {ALGORITHM}")
print(f"Samples: {SAMPLES}")
for model_run in MODEL_RUNS:
    print(f"- {model_run['display_name']}: {model_run['model_name']} ({model_run['quantization']})")


In [ ]:
# Clone the adapted repository into /content.
from pathlib import Path
import shutil
import subprocess

repo_path = Path(REPO_DIR)
if repo_path.exists():
    shutil.rmtree(repo_path)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
    check=True,
)
print("Cloned repository to:", REPO_DIR)


In [ ]:
# Install requirements.
subprocess.run(
    ["pip", "install", "-r", f"{REPO_DIR}/requirements.txt"],
    check=True,
)
print("Requirements installed.")


In [ ]:
# Verify actual model-file download access before starting the experiment.
from google.colab import userdata
from huggingface_hub import hf_hub_download, login

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Using the HF_TOKEN Colab Secret.")
else:
    print("No HF_TOKEN found. Gated Llama and Gemma downloads will fail.")

for model_run in MODEL_RUNS:
    model_name = model_run["model_name"]
    try:
        hf_hub_download(model_name, "config.json", token=hf_token)
        print("Model download access OK:", model_name)
    except Exception:
        print(f"Cannot download {model_name}.")
        print("Accept that model's Hugging Face terms using the account behind HF_TOKEN, then update the token.")
        raise


In [ ]:
# Start clean and save the model-run configuration.
import json
import os

if RESET_OUTPUTS and Path(OUTPUT_ROOT).exists():
    shutil.rmtree(OUTPUT_ROOT)
    print("Removed old outputs so every model uses the same fresh data.")

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

for model_run in MODEL_RUNS:
    model_run["attack_path"] = (
        f"{OUTPUT_ROOT}/sira_models/{model_run['label']}/final/{ALGORITHM}_attack.json"
    )

models_config_path = f"{OUTPUT_ROOT}/model_runs.json"
Path(models_config_path).write_text(json.dumps(MODEL_RUNS, indent=2), encoding="utf-8")

env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
env["ALGORITHM"] = ALGORITHM
env["SAMPLES"] = str(SAMPLES)

subprocess.run(
    ["python", "scripts/write_environment.py", "--output_path", f"{OUTPUT_ROOT}/environment.json", "--models_config", models_config_path],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


In [ ]:
# Run the identical SIRA workflow once for each attack model.
for model_run in MODEL_RUNS:
    model_env = env.copy()
    model_env["MODEL_NAME"] = model_run["model_name"]
    model_env["MODEL_LABEL"] = model_run["label"]
    model_env["LOAD_IN_4BIT"] = "true" if model_run["load_in_4bit"] else "false"

    print("\n" + "=" * 80)
    print("Running:", model_run["display_name"])
    print("=" * 80)

    try:
        subprocess.run(
            ["bash", "scripts/run_sira_model_l4.sh"],
            cwd=REPO_DIR,
            env=model_env,
            check=True,
        )
    except subprocess.CalledProcessError:
        log_path = Path(OUTPUT_ROOT) / "logs" / f"sira_{model_run['label']}.log"
        if log_path.exists():
            print("\nLast 100 log lines from", log_path)
            print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-100:]))
        raise

# Run the proposed 2x2 Cognitive Integrity grid-masking comparison.
subprocess.run(
    [
        "python", "scripts/run_cognitive_integrity_baseline.py",
        "--input_path", f"{OUTPUT_ROOT}/watermarked/{ALGORITHM}_response.json",
        "--output_path", f"{OUTPUT_ROOT}/cognitive_integrity/cognitive_integrity_attack.jsonl",
        "--model_name", "meta-llama/Llama-3.2-3B-Instruct",
        "--grid_size", "2",
        "--dtype", "bf16",
        "--max_samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


In [ ]:
# Evaluate attack success and semantic preservation across model families.
subprocess.run(
    [
        "python", "scripts/evaluate_sira_transfer.py",
        "--generation_model", "facebook/opt-1.3b",
        "--algorithm", ALGORITHM,
        "--watermarked_input", f"{OUTPUT_ROOT}/watermarked/{ALGORITHM}_response.json",
        "--models_config", models_config_path,
        "--cognitive_input", f"{OUTPUT_ROOT}/cognitive_integrity/cognitive_integrity_attack.jsonl",
        "--cognitive_grid_size", "2",
        "--output_root", OUTPUT_ROOT,
        "--dtype", "bf16",
        "--max_samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

subprocess.run(
    [
        "python", "scripts/compare_transfer_results.py",
        "--output_root", OUTPUT_ROOT,
        "--algorithm", ALGORITHM,
        "--samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


In [ ]:
# Display the final report and paper-style comparison dataframe.
import pandas as pd
from IPython.display import display

report_path = Path(OUTPUT_ROOT) / "final_report.md"
print(report_path.read_text(encoding="utf-8"))

comparison_dataframe = pd.read_csv(f"{OUTPUT_ROOT}/results/paper_style_comparison.csv")
display(comparison_dataframe)

print("\nSaved output files:")
for path in sorted(Path(OUTPUT_ROOT).rglob("*")):
    if path.is_file():
        print(path)


In [ ]:
# Copy results to Google Drive so they survive after the Colab runtime stops.
from google.colab import drive

drive.mount("/content/drive")
drive_output = Path("/content/drive/MyDrive/sira_transfer_outputs")

if drive_output.exists():
    shutil.rmtree(drive_output)

shutil.copytree(OUTPUT_ROOT, drive_output)
print("Copied results to:", drive_output)


## Stop the Runtime

After the Drive copy finishes, use **Runtime > Disconnect and delete runtime** to release the L4 GPU.
